# Tái thực nghiệm Decorrelated Variable Importance

Notebook này cài đặt phương pháp của Verdinelli và Wasserman, tái thực nghiệm
năm kịch bản mô phỏng của bài báo, và phân tích hai bộ dữ liệu chuẩn Energy Efficiency và Concrete
Compressive Strength. Các phương pháp đối chứng gồm LOCO, residual CPI và
marginal PFI.

Thiết kế thực nghiệm phục vụ trực tiếp các yêu cầu của đồ án:

1. Mô tả và kiểm chứng hiện tượng sai lệch do tương quan trong Kịch bản mô phỏng 1;
2. Đối chiếu độ lệch, RMSE, coverage và độ rộng khoảng trong các Kịch bản mô phỏng 2–5;
3. Đánh giá ít nhất hai baseline trên dữ liệu mô phỏng có giá trị mục tiêu đã biết;
4. Thực hiện ablation cho số hạng hiệu chỉnh của t-Cross;
5. Phân tích điều kiện phương pháp hoạt động ổn định hoặc trở nên bất ổn thông qua khả năng định danh, overlap, ESS của tỷ số mật độ và độ nhạy theo nuisance learner;
6. Ap dụng cùng quy trình trên hai bộ dữ liệu thực và lưu đầy đủ bảng, hình, metadata môi trường, checksum dữ liệu và hướng dẫn thực thi.

Notebook cung cấp ba cấu hình `smoke`, `reduced` và `paper`. Cấu hình `reduced`
giữ nguyên cấu trúc phương pháp nhưng giảm số lần lặp, số cây và số mẫu Monte
Carlo. Kết quả từ cấu hình này được báo cáo là **tái thực nghiệm giảm tải** và
không được diễn giải như bản sao chính xác của toàn bộ thí nghiệm trong bài báo.

Mọi bảng kết quả được xuất dưới dạng CSV. Mỗi hình được lưu đồng thời ở định
dạng PNG 300 DPI và PDF vector để sử dụng trong báo cáo.

## Cấu trúc mã nguồn

Toàn bộ mã phương pháp nằm trong thư mục `R/` của repository; notebook này chỉ
nạp module và gọi từng bước:

| Tệp | Nội dung |
| --- | --- |
| `R/config.R` | thư mục gốc, đường dẫn dữ liệu, giới hạn luồng, thứ tự nạp module |
| `R/constants.R` | hằng số, bảng màu, `get_profile`, coverage tham chiếu của Table 2 |
| `R/utils.R` | tiện ích số học, chia fold, basis đa thức trực giao, KDE, thống kê an toàn |
| `R/models.R` | ba nuisance learner: linear, `mgcv::gam`, `grf::regression_forest` |
| `R/estimators.R` | $\psi_L,\psi_0,\psi_1,\psi_2,\psi_3$, residual CPI và marginal PFI theo fold |
| `R/inference.R` | tổng hợp fold và thủ tục t-Cross |
| `R/simulation.R` | DGP Kịch bản 1–5, checkpoint, mô phỏng chính, benchmark, ablation |
| `R/real_data.R` | nạp, kiểm tra và phân tích Energy và Concrete |
| `R/plots.R` | toàn bộ hình dùng trong báo cáo |
| `R/reporting.R` | metadata môi trường, hướng dẫn thực thi, manifest đầu ra |
| `R/pipeline.R` | các stage và hàm điều phối `run_pipeline` |

Cùng bộ module đó được `main.R` sử dụng khi chạy không tương tác:

```
Rscript main.R --profile reduced --threads 8
Rscript main.R --profile paper --threads 16 --stages simulations,baseline
```

Script Slurm tương ứng nằm trong `jobs/reproduce/` và `jobs/stages/`.

## 1. Cấu hình thực nghiệm

Cell bên dưới đặt thư mục gốc, profile, số luồng rồi nạp toàn bộ module trong
`R/`. Mặc định notebook đọc `data/ENB2012_data.xlsx` và `data/Concrete_Data.xls`
theo đường dẫn tương đối so với `PROJECT_ROOT` (thư mục gốc đồ án; khi mở
notebook từ `src/` thì thư mục cha được dùng). Khi dữ liệu nằm nơi khác, ví dụ
trên Kaggle, truyền đường dẫn tuyệt đối trong bảng **Data** vào
`resolve_data_paths(energy = ..., concrete = ...)` hoặc đặt biến môi trường
`DVI_ENERGY_DATA_PATH` và `DVI_CONCRETE_DATA_PATH`.

Giới hạn luồng CPU phải được đặt trước khi nạp module vì `R/constants.R` và
`R/models.R` đọc biến môi trường ngay tại thời điểm `source`. Hàm
`apply_thread_limits` thực hiện việc này.

Từng bước thực nghiệm là một stage độc lập; chạy chọn lọc bằng cách chỉ thực thi
những cell tương ứng, hoặc bằng `--stages` khi dùng `main.R`.

Profile `reduced` sử dụng 10 lần lặp cho mô phỏng chính, 10 lần lặp cho benchmark
baseline và 30 lần lặp cho ablation. Coverage từ 10 lần lặp vẫn được xem là mô
tả thăm dò; kết luận về xác suất bao phủ phải được đối chiếu với khoảng tin cậy
nhị thức và kết quả 100 lần lặp trong bài báo.

Các checkpoint và tệp kết quả được lưu dưới `checkpoints/`, `results/` và
`figures/` trong `PROJECT_ROOT`. Mô phỏng chính ghi kết quả tăng dần theo tổ hợp
(kịch bản, tương quan, lần lặp, learner) nên một lần chạy bị ngắt có thể tiếp
tục mà không phải tính lại từ đầu.

In [ ]:
# Nạp cấu hình và toàn bộ module thực nghiệm.
# Mã phương pháp nằm trong thư mục `R/` của repository; notebook chỉ đóng vai trò
# giao diện trình bày. Cùng bộ mã đó được `main.R` dùng khi chạy bằng Rscript
# hoặc qua Slurm, nên kết quả của hai đường chạy là như nhau.

# Dò `R/config.R` từ thư mục làm việc lên tối đa ba cấp, nhờ vậy notebook chạy
# được cả khi kernel khởi động ở `src/` lẫn ở thư mục gốc của repository.
find_module_dir <- function(start = getwd()) {
  candidate <- normalizePath(start, winslash = "/", mustWork = TRUE)
  for (level in 0:3) {
    if (file.exists(file.path(candidate, "R", "config.R"))) {
      return(file.path(candidate, "R"))
    }
    parent <- dirname(candidate)
    if (identical(parent, candidate)) break
    candidate <- parent
  }
  stop("Không tìm thấy thư mục R/ chứa config.R; cần đặt DVI_PROJECT_ROOT.")
}

# Profile: "smoke", "reduced" hoặc "paper".
DVI_PROFILE <- "reduced"
N_THREADS <- 4L
REPORT_DPI_SETTING <- 300L
Sys.setenv(DVI_PROFILE = DVI_PROFILE, DVI_REPORT_DPI = REPORT_DPI_SETTING)

MODULE_DIR <- find_module_dir()
source(file.path(MODULE_DIR, "config.R"), encoding = "UTF-8")

N_THREADS <- apply_thread_limits(N_THREADS)
PROJECT_ROOT <- detect_project_root(dirname(MODULE_DIR))

# Mặc định đọc data/ENB2012_data.xlsx và data/Concrete_Data.xls theo PROJECT_ROOT.
# Trên Kaggle, truyền đường dẫn tuyệt đối trong bảng Data vào hai đối số dưới đây.
DATA_PATHS <- resolve_data_paths(energy = NULL, concrete = NULL)

load_dvi_modules(MODULE_DIR)
state <- new_pipeline_state(PROJECT_ROOT, DATA_PATHS, language = "R")
cat("Project root:", PROJECT_ROOT, "| profile:", DVI_PROFILE,
    "| threads:", N_THREADS, "\n")

## 2. Phương pháp và quy ước suy luận

Với $Y$ là biến đáp ứng, $X$ là biến được đánh giá và $Z$ là các biến còn lại,
LOCO được viết dưới dạng

$$
\psi_L
=\mathbb{E}\left[(Y-\mu(Z))^2\right]
-\mathbb{E}\left[(Y-\mu(X,Z))^2\right].
$$

Tham số decorrelated $\psi_0$ thay phân phối chung của $(X,Z)$ bằng tích các
phân phối biên trong thành phần rủi ro thứ hai. Các tham số $\psi_1,\psi_2$ và
$\psi_3$ là những xấp xỉ có cấu trúc khác nhau được trình bày trong bài báo.
Notebook sử dụng cross-fitting với năm fold và t-inference theo thủ tục t-Cross.

Đối với phần mô phỏng chính và dữ liệu thực, khoảng suy luận được trình bày
chính là khoảng t-Cross không có số hạng hiệu chỉnh bổ sung:

$$
\operatorname{se}_{\mathrm{cross}}^2=s^2/B.
$$

Lựa chọn này giữ cho độ rộng khoảng có thể diễn giải trong profile giảm tải.
Khoảng theo biểu thức

$$
\operatorname{se}^2=s^2/B+c^2/n
$$

vẫn được tính và lưu trong kết quả. Phần ablation đánh giá ba chế độ: không hiệu
chỉnh, $c=\operatorname{Var}(Y)$ và cách diễn giải trực tiếp
$c=\operatorname{Var}(Y)^2$. Coverage luôn được báo kèm độ rộng khoảng để tránh
kết luận dựa riêng trên xác suất bao phủ.

Các baseline có ý nghĩa khác nhau:

- `psi_L` là LOCO và có thể chịu sai lệch do tương quan giữa $X$ và $Z$;
- `cpi_residual` là xấp xỉ Conditional Permutation Importance bằng hoán vị phần
  dư của mô hình $X\mid Z$;
- `pfi` là marginal permutation importance, cố ý phá vỡ phân phối chung của
  $(X,Z)$ và không phải bộ ước lượng trực tiếp của $\psi_0$.

Vì vậy, benchmark baseline báo độ lệch, RMSE và tỷ lệ khoảng chứa oracle
$\psi_0$ như các chẩn đoán thực nghiệm. Các đại lượng này không được diễn giải
như coverage danh nghĩa của cùng một estimand.


### 2.1. Mô hình nuisance

Ba nuisance learner được sử dụng nhất quán trong các thí nghiệm: hồi quy tuyến
tính, mô hình cộng tính `mgcv::gam`, và rừng hồi quy
`grf::regression_forest`. Cùng seed, số fold và giới hạn luồng tính toán được
dùng giữa các phương pháp để giảm sai khác không liên quan đến estimand.

Cài đặt: `R/models.R` (`fit_regressor`, `fit_additive_gam`,
`fit_multivariate_regression`).

### 2.2. Bộ ước lượng, KDE và t-Cross

Cài đặt: `R/utils.R` cho basis đa thức trực giao và KDE Gaussian chuẩn hóa;
`R/estimators.R` cho $\psi_L,\psi_0,\psi_1,\psi_2,\psi_3$ cùng hai baseline
residual CPI và marginal PFI trên từng fold; `R/inference.R` cho bước tổng hợp
fold và khoảng t-Cross.

### 2.3. DGP, checkpoint, dữ liệu thực và trực quan hóa

Cài đặt: `R/simulation.R` cho năm DGP, giá trị mục tiêu giải tích, checkpoint và
ba vòng chạy (mô phỏng chính, benchmark baseline, ablation); `R/real_data.R` cho
Energy và Concrete; `R/plots.R` cho toàn bộ hình; `R/reporting.R` cho metadata
môi trường, hướng dẫn thực thi và manifest.

### Quy ước ký hiệu và tên kịch bản trong hình

Các hình sử dụng biểu thức toán học của R để hiển thị
$\psi_L,\psi_0,\psi_1,\psi_2,\psi_3$, $R^2$ và
$\operatorname{Var}(Y)$. Chuỗi `psi_0` chỉ được giữ làm tên biến nội bộ trong
mã và tên cột dữ liệu; nó không được dùng làm nhãn hiển thị.

Năm thiết kế sinh dữ liệu của bài báo được gọi thống nhất là
**Kịch bản mô phỏng 1–5**. Tên này phân biệt chúng với các ví dụ minh họa thông
thường và phù hợp với cách trình bày trong báo cáo.

## 3. Kiểm tra môi trường, cấu hình và tính toàn vẹn dữ liệu

Stage `check_environment` kiểm tra package, kích thước dữ liệu, biến đáp ứng,
biến được đánh giá, tập biến điều kiện và SHA-256 của tệp đầu vào. Việc kiểm tra
được thực hiện trước mọi tác vụ tốn thời gian để tránh tạo kết quả từ dữ liệu
hoặc môi trường không đúng cấu hình.

In [ ]:
state <- stage_check_environment(state, DVI_PROFILE, N_THREADS)
config <- state$config

## 4. Giá trị mục tiêu giải tích

Bảng dưới đây cung cấp các target dùng để đánh giá độ lệch và coverage trên dữ
liệu mô phỏng. Hình của Kịch bản mô phỏng 1 minh họa trực tiếp sự suy giảm của $\psi_L$ khi
tương quan tăng, trong khi target decorrelated $\psi_0$ giữ nguyên.

In [ ]:
state <- stage_targets(state)

## 5. Mô phỏng chính và phân tích điều kiện hoạt động

Mô phỏng chính tái tạo các Kịch bản mô phỏng 1–5. Đối với mỗi tổ hợp, bảng tổng hợp báo:

- trung bình, độ lệch chuẩn thực nghiệm và RMSE của ước lượng;
- độ lệch tuyệt đối và tương đối so với oracle $\psi_0$;
- coverage t-Cross không hiệu chỉnh cùng khoảng tin cậy nhị thức chính xác;
- coverage tham chiếu từ Table 2 và chênh lệch so với bài báo;
- độ rộng khoảng, tỷ lệ ước lượng hữu hạn và ESS của tỷ số mật độ;
- cờ chẩn đoán định lượng về độ lệch, coverage, độ rộng khoảng, overlap và số
  lần lặp.

Với profile `reduced`, coverage của mô phỏng chính vẫn là kết quả thăm dò. Các
figure được thiết kế để tách hành vi của $\psi_0$ khỏi các estimator ổn định
hơn, đồng thời liên hệ sai lệch của $\psi_0$ với ESS của tỷ số mật độ.


In [ ]:
state <- stage_simulations(state)

## 6. Benchmark baseline có oracle

Benchmark sử dụng các Kịch bản mô phỏng 1, 2 và 5, là các trường hợp có $\psi_0$ đã biết.
LOCO, residual CPI và marginal PFI được chạy với cùng dữ liệu, fold, learner và
seed. DVI $\psi_0$ từ mô phỏng chính được ghép vào bảng so sánh.

Đánh giá baseline sử dụng đồng thời:

- độ lệch tương đối tuyệt đối;
- RMSE tương đối so với oracle $\psi_0$;
- độ lệch chuẩn giữa các lần lặp;
- tỷ lệ khoảng t-Cross chứa oracle và khoảng tin cậy nhị thức tương ứng;
- tỷ lệ ước lượng hữu hạn.

Marginal PFI không cùng estimand với $\psi_0$; vị trí của phương pháp này trong
bảng và hình chỉ mô tả hệ quả của phép nhiễu biên. Tỷ lệ khoảng chứa oracle của
các baseline là một chẩn đoán so sánh, không phải coverage danh nghĩa của cùng
một tham số.


In [ ]:
state <- stage_baseline(state)

## 7. Ablation của số hạng hiệu chỉnh t-Cross

Ablation dùng Kịch bản mô phỏng 2, nuisance learner tuyến tính và cùng dữ liệu/fold giữa ba
chế độ hiệu chỉnh trong từng lần lặp. Một lần ước lượng fold được tái sử dụng để
tránh huấn luyện lại khi chỉ thay đổi công thức tổng hợp khoảng tin cậy.

Kết quả gồm coverage với khoảng tin cậy nhị thức chính xác, độ rộng trung bình,
độ rộng trung vị, phân vị 90%, hệ số phóng đại độ rộng so với chế độ không hiệu
chỉnh và mức thay đổi coverage. Thiết kế này đánh giá đồng thời lợi ích coverage
và chi phí về độ rộng khoảng.


In [ ]:
state <- stage_ablation(state)

## 8. Phân tích hai bộ dữ liệu thực

Việc chọn $X$ chỉ dựa trên quan hệ giữa các biến dự báo, không sử dụng biến đáp
ứng để lựa chọn kịch bản. Energy dùng `relative_compactness` và loại
`roof_area` khỏi $Z$ do quan hệ đại số với `surface_area` và `wall_area`.
Concrete dùng `water` và bảy biến dự báo còn lại làm $Z$.

Không có oracle $\psi_0$ trên dữ liệu thực. Vì vậy, phân tích tập trung vào:

- khả năng dự đoán chéo của $X$ từ $Z$;
- tỷ lệ phương sai phần dư và ESS của tỷ số mật độ;
- trạng thái định danh và tỷ lệ fold hữu hạn;
- độ nhạy của ước lượng theo nuisance learner;
- mức chênh lệch giữa DVI, LOCO, residual CPI và marginal PFI.

Hình importance sử dụng khoảng t-Cross không hiệu chỉnh và chuẩn hóa theo
$\operatorname{Var}(Y)$ để giữ khả năng đọc và dùng chung thang ngang giữa các
learner. Khoảng có số hạng hiệu chỉnh vẫn được lưu trong bảng CSV để phục vụ
phân tích độ nhạy.


In [ ]:
state <- stage_real_data(state)

## 9. Nguyên tắc diễn giải và giới hạn

Các kết luận trong báo cáo cần tuân theo các quy ước sau:

- Profile `reduced` kiểm tra hành vi định tính và khả năng chạy của pipeline;
  coverage với ít hơn 30 lần lặp chỉ mang tính mô tả.
- Độ lệch và RMSE với oracle chỉ được tính trên dữ liệu mô phỏng. Dữ liệu thực
  chỉ hỗ trợ so sánh mô tả và chẩn đoán điều kiện định danh.
- $R^2(X\mid Z)$ cao, tỷ lệ phương sai phần dư nhỏ hoặc ESS của tỷ số mật độ
  thấp cho thấy overlap hiệu dụng yếu. Trong trường hợp này, sai khác lớn giữa
  learner là dấu hiệu bất ổn, không phải bằng chứng về importance lớn.
- Residual CPI phụ thuộc vào chất lượng mô hình $X\mid Z$. Marginal PFI phá vỡ
  phân phối chung của $(X,Z)$ và không được diễn giải như estimator của
  $\psi_0$.
- Coverage phải được đọc cùng khoảng tin cậy nhị thức và độ rộng khoảng.
  Một chế độ có coverage cao nhưng độ rộng tăng nhiều lần không được xem là
  vượt trội chỉ dựa trên coverage.
- Mọi trường hợp không định danh, không hữu hạn hoặc có cảnh báo overlap phải
  được giữ lại trong bảng và thảo luận.
- Các cột `diagnostic` sử dụng ngưỡng được công bố trực tiếp trong mã. Chúng là
  công cụ hỗ trợ đọc kết quả và không thay thế phân tích theo từng kịch bản.


In [ ]:
state <- stage_reporting(state)

## 10. Đối chiếu với yêu cầu thực nghiệm của đồ án

Bảng cuối kiểm tra trực tiếp các đầu ra bắt buộc: hai bộ dữ liệu chuẩn, ít nhất
hai baseline, độ đo phù hợp, bảng và hình, phân tích điều kiện hoạt động,
ablation và tài liệu tái thực nghiệm. Trạng thái được suy ra từ các đối tượng và
tệp đã tạo trong lần chạy hiện tại.


In [ ]:
state <- stage_compliance(state)